# LeetCode 3222 — Find the Winning Player in Coin Game bằng Python

Link đề: https://leetcode.com/problems/find-the-winning-player-in-coin-game/ · Độ khó: **Easy** · Chủ đề: Math, Game Theory, Simulation

Skeleton LeetCode:

```python
class Solution:
    def losingPlayer(self, x: int, y: int) -> str:
        ...
```

Ghi chú: tên hàm trên LeetCode là `losingPlayer` (người thua) nhưng đề yêu cầu **trả về tên người THẮNG** — đây là một "bẫy tên gọi" nổi tiếng của bài này, notebook sẽ nhắc lại ở phần lỗi thường gặp. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn toàn bộ `1 <= x, y <= 100`). Các ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `x` đồng xu giá trị `75` và `y` đồng xu giá trị `10`; Alice và Bob lần lượt (Alice đi trước) phải nhặt các đồng xu có **tổng giá trị đúng `115`**; ai **không nhặt được** thì **thua**; trả về tên người **thắng** khi cả hai chơi tối ưu.
2. Nắm các công cụ Python: phép chia nguyên `//`, `min`, vòng lặp `while`, kiểm tra chẵn/lẻ bằng `% 2` hoặc `& 1`, biểu thức điều kiện `"Alice" if ... else "Bob"`, và đệ quy có nhớ (`functools.lru_cache`) để vét cạn trò chơi.
3. Đi từ cách **minimax vét cạn tổng quát** → **mô phỏng từng lượt** → **đếm số lượt bằng công thức** → **gộp hai lượt thành một "vòng"** (cách của doocs) → **một dòng bằng chẵn/lẻ**.
4. Hiểu mấu chốt: tổng `115` chỉ tạo được bằng **đúng một cách** là `1` xu `75` + `4` xu `10` (`75 + 40 = 115`), nên **không có chiến thuật nào để chọn** — trò chơi hoàn toàn xác định, chỉ cần biết **số lượt tối đa** là chẵn hay lẻ.
5. Tránh các lỗi kinh điển: trả về người thua thay vì người thắng (bẫy tên hàm), quên điều kiện `y >= 4` hoặc `x >= 1`, đảo ngược chẵn/lẻ, và lấy số lượt chỉ theo `x` hoặc chỉ theo `y`.

## 2. Đề bài & ví dụ

Cho hai số nguyên **dương** `x` và `y`: số đồng xu giá trị `75` và số đồng xu giá trị `10`.

Alice và Bob chơi một trò chơi. **Mỗi lượt**, người chơi (bắt đầu từ **Alice**, sau đó đến Bob, rồi lại Alice...) phải nhặt các đồng xu có **tổng giá trị đúng `115`**. Nếu không thể làm điều đó thì người đó **thua**.

Hãy trả về **tên** người thắng nếu cả hai chơi **tối ưu**.

**Ví dụ 1**

- Input: `x = 2, y = 7`
- Output: `"Alice"`
- Giải thích: trò chơi kết thúc sau một lượt. Alice nhặt `1` xu `75` và `4` xu `10` (`75 + 40 = 115`). Còn lại `1` xu `75` và `3` xu `10`: Bob không thể nhặt đủ `115` nên Bob thua.

**Ví dụ 2**

- Input: `x = 4, y = 11`
- Output: `"Bob"`
- Giải thích: trò chơi kéo dài hai lượt. Alice nhặt `1` xu `75` + `4` xu `10`; Bob nhặt `1` xu `75` + `4` xu `10`. Còn `2` xu `75` và `3` xu `10`: Alice không nhặt được nên Alice thua, Bob thắng.

**Ràng buộc:** `1 <= x, y <= 100`.

In [1]:
examples = [
    (2, 7, "Alice"),
    (4, 11, "Bob"),
]
for x, y, expected in examples:
    print(f"x={x}, y={y} -> kỳ vọng {expected!r}")

x=2, y=7 -> kỳ vọng 'Alice'
x=4, y=11 -> kỳ vọng 'Bob'


## 3. Kiến thức nền

### 3.1. `//` (chia nguyên), `%` (chia dư) và `min`

`a // b` là thương **làm tròn xuống** (`7 // 4 = 1`), `a % b` là phần dư (`7 % 4 = 3`). Trong bài này `y // 4` cho biết **tối đa bao nhiêu lần** ta có thể lấy bộ `4` xu `10` từ `y` xu. `min(a, b)` lấy số nhỏ hơn — dùng khi một việc cần **cả hai** nguồn lực (ở đây: đủ xu `75` **và** đủ xu `10`).

In [2]:
print(7 // 4, 7 % 4)              # 1 3
print(11 // 4, 11 % 4)            # 2 3
print(min(4, 11 // 4))            # 2  -> bị "nút cổ chai" bởi y
print(min(2, 7 // 4))             # 1

1 3
2 3
2
1


### 3.2. Vòng lặp `while` và đổi lượt

`while điều_kiện:` lặp **chừng nào** điều kiện còn đúng. Để mô phỏng hai người luân phiên ta dùng một biến đánh dấu, ví dụ `alice_turn = True`, rồi đảo bằng `alice_turn = not alice_turn`. Khi vòng lặp dừng, người đang **đến lượt** mà không đi được chính là người thua.

In [3]:
alice_turn = True
for step in range(5):
    print("lượt", step + 1, "->", "Alice" if alice_turn else "Bob")
    alice_turn = not alice_turn

lượt 1 -> Alice
lượt 2 -> Bob
lượt 3 -> Alice
lượt 4 -> Bob
lượt 5 -> Alice


### 3.3. Chẵn / lẻ: `% 2` và `& 1`

Một số `t` là **lẻ** khi `t % 2 == 1` (hay `t & 1 == 1`: bit thấp nhất bằng `1`). Trong trò chơi hai người luân phiên: nếu có **tổng cộng `t` lượt đi được** thì người đi lượt thứ `t` là Alice khi `t` **lẻ** (Alice đi lượt 1, 3, 5, ...) và là Bob khi `t` **chẵn** (và `t = 0` nghĩa là Alice không đi nổi lượt đầu).

In [4]:
for t in range(6):
    print(f"t={t}: t%2={t % 2}, t&1={t & 1}, người đi lượt cuối:", "Alice" if t % 2 else ("Bob" if t else "không ai"))

t=0: t%2=0, t&1=0, người đi lượt cuối: không ai
t=1: t%2=1, t&1=1, người đi lượt cuối: Alice
t=2: t%2=0, t&1=0, người đi lượt cuối: Bob
t=3: t%2=1, t&1=1, người đi lượt cuối: Alice
t=4: t%2=0, t&1=0, người đi lượt cuối: Bob
t=5: t%2=1, t&1=1, người đi lượt cuối: Alice


### 3.4. Biểu thức điều kiện (toán tử ba ngôi)

`A if điều_kiện else B` là cách viết ngắn của `if/else` trả về một giá trị. Ví dụ `"Alice" if t % 2 else "Bob"`: nếu `t % 2` khác `0` (lẻ) trả `"Alice"`, ngược lại trả `"Bob"`. Số `0` được coi là "sai", mọi số khác `0` là "đúng".

In [5]:
t = 3
print("Alice" if t % 2 else "Bob")      # Alice
t = 4
print("Alice" if t % 2 else "Bob")      # Bob
print(bool(0), bool(5))                 # False True

Alice
Bob
False True


### 3.5. Vét cạn các cách lấy tổng `115`

Để chắc chắn không bỏ sót, ta thử mọi cặp `(a, b)`: lấy `a` xu `75` và `b` xu `10` sao cho `75a + 10b = 115`. Vì `75a <= 115` nên `a` chỉ có thể là `0` hoặc `1`; `b` nhiều nhất `11`.

In [6]:
ways = []
for a in range(0, 115 // 75 + 1):          # a = số xu 75
    for b in range(0, 115 // 10 + 1):      # b = số xu 10
        if 75 * a + 10 * b == 115:
            ways.append((a, b))
print(ways)                                # [(1, 4)]: chỉ MỘT cách duy nhất

[(1, 4)]


### 3.6. Đệ quy có nhớ: `functools.lru_cache`

Khi một hàm đệ quy gọi lại cùng tham số nhiều lần, `@lru_cache(maxsize=None)` ghi nhớ kết quả để mỗi trạng thái chỉ tính **một lần**. Hàm phải nhận tham số "băm được" (số nguyên, tuple...). Ta sẽ dùng nó để vét cạn trò chơi ở Cách 1.

In [7]:
from functools import lru_cache

calls = 0

@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "số lần tính thật:", calls)    # 832040 31 (không cache thì ~2.7 triệu lần)

832040 số lần tính thật: 31


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Vì mỗi lượt **bắt buộc** lấy đúng `1` xu `75` và `4` xu `10`, trạng thái chỉ đi theo một đường: `(x, y) → (x − 1, y − 4)`. Ai tới lượt mà `x = 0` **hoặc** `y < 4` thì thua.

Ví dụ 2: `x = 4`, `y = 11`:

| lượt | người đi | trước lượt `(x, y)` | đi được không? | sau lượt `(x, y)` |
|---|---|---|---|---|
| 1 | Alice | (4, 11) | có (`x ≥ 1`, `y ≥ 4`) | (3, 7) |
| 2 | Bob | (3, 7) | có | (2, 3) |
| 3 | Alice | (2, 3) | **không** (`y = 3 < 4`) | — Alice thua |

Tổng cộng **2 lượt** đi được (số chẵn) ⇒ lượt cuối là của Bob ⇒ **Bob thắng**. Trực giác: ta chỉ cần biết trò chơi kéo dài **tối đa bao nhiêu lượt** `T`, rồi nhìn `T` lẻ hay chẵn.

In [8]:
x, y = 4, 11
alice = True
while x >= 1 and y >= 4:
    print(("Alice" if alice else "Bob"), "đi:", (x, y), "->", (x - 1, y - 4))
    x, y = x - 1, y - 4
    alice = not alice
print("Người không đi được:", "Alice" if alice else "Bob")

Alice đi: (4, 11) -> (3, 7)
Bob đi: (3, 7) -> (2, 3)
Người không đi được: Alice


## 5. Cách 1 — Brute Force / baseline: minimax vét cạn tổng quát

**Ý tưởng.** Không giả định gì về "chỉ có một cách lấy". Với trạng thái `(x, y)` và người sắp đi:

- Liệt kê **mọi** cặp `(a, b)` với `a <= x`, `b <= y`, `75a + 10b = 115`.
- Người đi **thắng** nếu **tồn tại** một nước đi dẫn tới trạng thái mà **đối thủ thua**.
- Nếu **không có nước đi nào** thì người đó thua.

`win(x, y)` trả `True` nếu người **đang tới lượt** thắng. Đáp án: `"Alice" if win(x, y) else "Bob"`. Đây đúng là định nghĩa "chơi tối ưu".

In [9]:
from functools import lru_cache


def winner_minimax(x: int, y: int) -> str:
    @lru_cache(maxsize=None)
    def win(x, y):                                    # người đang tới lượt có thắng không?
        for a in range(0, min(x, 1) + 1):             # a xu 75 (75*2 > 115 nên a <= 1)
            for b in range(0, min(y, 11) + 1):        # b xu 10
                if 75 * a + 10 * b == 115:
                    if not win(x - a, y - b):         # tồn tại nước dẫn tới đối thủ thua
                        return True
        return False                                  # hết nước đi hoặc nước nào cũng thua

    return "Alice" if win(x, y) else "Bob"


for x, y, expected in examples:
    print(winner_minimax(x, y), "(kỳ vọng", expected, ")")

Alice (kỳ vọng Alice )
Bob (kỳ vọng Bob )


**Độ phức tạp (Cách 1).** Số trạng thái `(x, y)` tối đa `100 × 100 = 10⁴`, mỗi trạng thái thử tối đa `2 × 12 = 24` cặp ⇒ **Time `O(x · y)`** (hằng số nhỏ), **Space `O(x · y)`** cho bảng nhớ và ngăn xếp đệ quy sâu tới `min(x, y // 4)`.

**Nút thắt:** ta đã chứng minh (mục 3.5) rằng chỉ có **một** nước đi hợp lệ, nên cả cây trò chơi là một đường thẳng — không cần minimax, chỉ cần **đếm độ dài** của đường đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ có một nước đi** (`1` xu `75` + `4` xu `10`), nên không ai "chọn" được gì: trò chơi là một dãy lượt cố định, kết thúc khi `x = 0` hoặc `y < 4`.

Số lượt đi được tối đa là

> `T = min(x, y // 4)`

(mỗi lượt tốn `1` xu `75` ⇒ không quá `x` lượt; mỗi lượt tốn `4` xu `10` ⇒ không quá `y // 4` lượt; cả hai giới hạn phải thỏa nên lấy `min`).

- `T` **lẻ** ⇒ Alice đi lượt cuối, Bob không đi được ⇒ **Alice thắng**.
- `T` **chẵn** (kể cả `0`) ⇒ người không đi được là Alice ⇒ **Bob thắng**.

Vậy đáp án là `"Alice" if min(x, y // 4) % 2 == 1 else "Bob"` — `O(1)`.

In [10]:
for x, y, expected in examples:
    T = min(x, y // 4)
    print(f"x={x}, y={y}: x={x}, y//4={y // 4} -> T={T}, {'lẻ' if T % 2 else 'chẵn'} ->", "Alice" if T % 2 else "Bob")

x=2, y=7: x=2, y//4=1 -> T=1, lẻ -> Alice
x=4, y=11: x=4, y//4=2 -> T=2, chẵn -> Bob


## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng từng lượt bằng `while`

**Giải thích từng đoạn.**

1. `alice_turn = True`: Alice đi trước.
2. `while x >= 1 and y >= 4:` — còn đủ `1` xu `75` **và** `4` xu `10` thì lượt hiện tại đi được.
3. Trừ `x -= 1`, `y -= 4`, rồi đảo lượt.
4. Thoát vòng lặp: người đang tới lượt **không đi được** ⇒ người đó thua ⇒ người kia thắng.

In [11]:
def winner_simulation(x: int, y: int) -> str:
    alice_turn = True
    while x >= 1 and y >= 4:        # còn đủ 1 xu 75 và 4 xu 10
        x -= 1
        y -= 4
        alice_turn = not alice_turn  # sang lượt người kia
    # alice_turn là người KHÔNG đi được -> người đó thua
    return "Bob" if alice_turn else "Alice"


for x, y, expected in examples:
    print(winner_simulation(x, y), "(kỳ vọng", expected, ")")

Alice (kỳ vọng Alice )
Bob (kỳ vọng Bob )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSimulation:
    def losingPlayer(self, x: int, y: int) -> str:
        alice_turn = True
        while x >= 1 and y >= 4:
            x -= 1
            y -= 4
            alice_turn = not alice_turn
        return "Bob" if alice_turn else "Alice"


print(SolutionSimulation().losingPlayer(2, 7), SolutionSimulation().losingPlayer(4, 11))    # Alice Bob

Alice Bob


**Độ phức tạp Cách 2:** số vòng lặp tối đa `min(x, y // 4) <= 25` ⇒ **Time `O(min(x, y / 4))`**, **Space `O(1)`**.

### Cách 3 — Công thức số lượt `T` rồi xét chẵn/lẻ

**Giải thích.** Thay vòng lặp bằng công thức đã suy ra ở mục 6: `T = min(x, y // 4)`. Rồi trả `"Alice"` nếu `T` lẻ. Đây là cách ngắn và dễ giải thích nhất.

In [13]:
def winner_parity(x: int, y: int) -> str:
    turns = min(x, y // 4)                  # số lượt tối đa đi được
    return "Alice" if turns % 2 == 1 else "Bob"


for x, y, expected in examples:
    print(winner_parity(x, y), "(kỳ vọng", expected, ")")

Alice (kỳ vọng Alice )
Bob (kỳ vọng Bob )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionParity:
    def losingPlayer(self, x: int, y: int) -> str:
        turns = min(x, y // 4)
        return "Alice" if turns % 2 == 1 else "Bob"


print(SolutionParity().losingPlayer(2, 7), SolutionParity().losingPlayer(4, 11))

Alice Bob


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Gộp hai lượt thành một "vòng" (cách của doocs)

**Giải thích.** Một **vòng** = Alice đi rồi Bob đi = lấy `2` xu `75` và `8` xu `10`. Số vòng trọn vẹn là `k = min(x // 2, y // 8)`. Sau `k` vòng, ai cũng đã đi `k` lượt nên đang là **lượt của Alice**; còn lại `x' = x − 2k`, `y' = y − 8k`. Alice đi được khi `x' > 0` và `y' >= 4`. Nếu Alice đi được thì Bob **chắc chắn** không đi được (vì nếu Bob cũng đi được thì còn thêm một vòng nữa, mâu thuẫn với `k` lớn nhất) ⇒ Alice thắng; ngược lại Bob thắng.

In [15]:
def winner_rounds(x: int, y: int) -> str:
    k = min(x // 2, y // 8)         # số "vòng" Alice+Bob trọn vẹn
    x -= 2 * k
    y -= 8 * k
    return "Alice" if x > 0 and y >= 4 else "Bob"


for x, y, expected in examples:
    print(winner_rounds(x, y), "(kỳ vọng", expected, ")")

Alice (kỳ vọng Alice )
Bob (kỳ vọng Bob )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionRounds:
    def losingPlayer(self, x: int, y: int) -> str:
        k = min(x // 2, y // 8)
        x -= k * 2
        y -= k * 8
        return "Alice" if x and y >= 4 else "Bob"


print(SolutionRounds().losingPlayer(2, 7), SolutionRounds().losingPlayer(4, 11))

Alice Bob


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 5 — Một dòng bằng bit thấp nhất (`& 1`) và chỉ số của chuỗi

**Giải thích.** `min(x, y // 4) & 1` là `1` khi lẻ, `0` khi chẵn. Ta dùng nó làm **chỉ số** vào bộ `("Bob", "Alice")`: chỉ số `0` ⇒ `"Bob"`, chỉ số `1` ⇒ `"Alice"`. Gọn nhưng khó đọc hơn Cách 3 — chỉ nên dùng khi bạn đã hiểu rõ lý do.

In [17]:
def winner_bit(x: int, y: int) -> str:
    return ("Bob", "Alice")[min(x, y // 4) & 1]


for x, y, expected in examples:
    print(winner_bit(x, y), "(kỳ vọng", expected, ")")

Alice (kỳ vọng Alice )
Bob (kỳ vọng Bob )


**Độ phức tạp Cách 5:** **Time `O(1)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (mô phỏng) trên `x = 4, y = 11`, Cách 3 (công thức) trên `x = 2, y = 7`, và Cách 4 (vòng) trên `x = 9, y = 30` — ví dụ đủ "khó" để có cả vòng trọn vẹn lẫn lượt lẻ ở cuối. (`x = 9, y = 30` là ví dụ tự dựng, đáp án đã kiểm bằng minimax.)

In [18]:
def trace_simulation(x: int, y: int) -> str:
    print(f"x = {x}, y = {y}")
    alice_turn = True
    step = 0
    while x >= 1 and y >= 4:
        step += 1
        who = "Alice" if alice_turn else "Bob"
        print(f"  lượt {step}: {who} nhặt 1 xu 75 + 4 xu 10 -> còn x={x - 1}, y={y - 4}")
        x -= 1
        y -= 4
        alice_turn = not alice_turn
    loser = "Alice" if alice_turn else "Bob"
    winner = "Bob" if alice_turn else "Alice"
    print(f"  tới lượt {loser} nhưng x={x}, y={y} không đủ -> {loser} thua, {winner} thắng")
    return winner


assert trace_simulation(4, 11) == "Bob"

x = 4, y = 11
  lượt 1: Alice nhặt 1 xu 75 + 4 xu 10 -> còn x=3, y=7
  lượt 2: Bob nhặt 1 xu 75 + 4 xu 10 -> còn x=2, y=3
  tới lượt Alice nhưng x=2, y=3 không đủ -> Alice thua, Bob thắng


In [19]:
def trace_parity(x: int, y: int) -> str:
    print(f"x = {x}, y = {y}")
    print(f"  x = {x}, y // 4 = {y // 4}")
    turns = min(x, y // 4)
    print(f"  T = min({x}, {y // 4}) = {turns}")
    print(f"  T % 2 = {turns % 2} ->", "lẻ: Alice đi lượt cuối" if turns % 2 else "chẵn: Bob đi lượt cuối (hoặc không ai đi)")
    return "Alice" if turns % 2 else "Bob"


assert trace_parity(2, 7) == "Alice"
assert trace_parity(1, 100) == "Alice"      # y thừa rất nhiều nhưng chỉ có 1 xu 75

x = 2, y = 7
  x = 2, y // 4 = 1
  T = min(2, 1) = 1
  T % 2 = 1 -> lẻ: Alice đi lượt cuối
x = 1, y = 100
  x = 1, y // 4 = 25
  T = min(1, 25) = 1
  T % 2 = 1 -> lẻ: Alice đi lượt cuối


In [20]:
def trace_rounds(x: int, y: int) -> str:
    print(f"x = {x}, y = {y}")
    k = min(x // 2, y // 8)
    print(f"  k = min({x // 2}, {y // 8}) = {k} vòng (mỗi vòng: -2 xu 75, -8 xu 10)")
    x -= 2 * k
    y -= 8 * k
    print(f"  sau {k} vòng: x = {x}, y = {y} (tới lượt Alice)")
    alice_ok = x > 0 and y >= 4
    print(f"  Alice đi được? x > 0: {x > 0}, y >= 4: {y >= 4} -> {alice_ok}")
    return "Alice" if alice_ok else "Bob"


assert trace_rounds(9, 30) == "Alice"
assert trace_rounds(9, 30) == winner_minimax(9, 30)

x = 9, y = 30
  k = min(4, 3) = 3 vòng (mỗi vòng: -2 xu 75, -8 xu 10)
  sau 3 vòng: x = 3, y = 6 (tới lượt Alice)
  Alice đi được? x > 0: True, y >= 4: True -> True
x = 9, y = 30
  k = min(4, 3) = 3 vòng (mỗi vòng: -2 xu 75, -8 xu 10)
  sau 3 vòng: x = 3, y = 6 (tới lượt Alice)
  Alice đi được? x > 0: True, y >= 4: True -> True


Bảng trace Cách 2 với `x = 4, y = 11`:

| lượt | người đi | `x` trước | `y` trước | `x` sau | `y` sau |
|---|---|---|---|---|---|
| 1 | Alice | 4 | 11 | 3 | 7 |
| 2 | Bob | 3 | 7 | 2 | 3 |
| 3 | Alice | 2 | 3 | không đi được (`y < 4`) | Alice thua |

`T = 2` (chẵn) ⇒ **Bob**. Với `x = 9, y = 30`: `T = min(9, 7) = 7` (lẻ) ⇒ **Alice**. Cách 4: `k = min(4, 3) = 3`, còn `x = 3, y = 6`, Alice đi được (`3 > 0`, `6 >= 4`) ⇒ **Alice** (khớp).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (đếm số lượt rồi xét chẵn/lẻ): ngắn, `O(1)`, dễ chứng minh. Nhớ: **hàm tên `losingPlayer` nhưng trả về tên người thắng**.

In [21]:
class Solution:
    def losingPlayer(self, x: int, y: int) -> str:
        turns = min(x, y // 4)                      # số lượt tối đa (mỗi lượt: 1 xu 75 + 4 xu 10)
        return "Alice" if turns % 2 == 1 else "Bob"  # lẻ: Alice đi lượt cuối -> Alice thắng


sol = Solution()
for x, y, expected in examples:
    print(x, y, "->", sol.losingPlayer(x, y))

2 7 -> Alice
4 11 -> Bob


## 10. Vì sao thuật toán đúng

- **Chỉ có một nước đi.** `75a + 10b = 115` với `a, b ≥ 0` nguyên: `a ≥ 2` cho `150 > 115` (loại); `a = 0` cho `10b = 115` (không nguyên, loại); `a = 1` cho `10b = 40`, `b = 4`. Vậy mỗi lượt đúng là lấy `(1, 4)`.
- **Trò chơi xác định.** Không có lựa chọn nên "chơi tối ưu" không thay đổi gì: sau `j` lượt, trạng thái là `(x − j, y − 4j)`.
- **Số lượt tối đa.** Lượt thứ `j` đi được khi `x − (j − 1) ≥ 1` và `y − 4(j − 1) ≥ 4`, tức `j ≤ x` và `j ≤ y // 4`. Vậy có đúng `T = min(x, y // 4)` lượt đi được, rồi dừng.
- **Ai thua?** Lượt thứ `T + 1` không đi được. Lượt số lẻ là của Alice, số chẵn là của Bob. Nếu `T` lẻ thì `T + 1` chẵn ⇒ Bob không đi được ⇒ **Alice thắng**; nếu `T` chẵn thì `T + 1` lẻ ⇒ Alice không đi được ⇒ **Bob thắng**.
- **Đối chiếu độc lập.** Cách 1 (minimax không giả định gì) được so với công thức trên toàn bộ `100 × 100` cặp ở mục 12.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Minimax có nhớ | `O(x · y)` (hằng số nhỏ) | `O(x · y)` |
| 2. Mô phỏng `while` | `O(min(x, y / 4))` | `O(1)` |
| 3. Công thức `T` + chẵn/lẻ | `O(1)` | `O(1)` |
| 4. Gộp vòng (doocs) | `O(1)` | `O(1)` |
| 5. Chỉ số chuỗi bằng bit | `O(1)` | `O(1)` |

Với `x, y ≤ 100` mọi cách đều chạy tức thì; điểm chính là **bài học mô hình hóa**: nhận ra trò chơi không có lựa chọn rồi quy về tính chẵn/lẻ.

## 12. Bộ test

Gồm: ví dụ đề, biên (`x = 1`, `y = 3` — thiếu xu 10, `y = 4` — vừa đủ), phản ví dụ cho những công thức sai, đối chiếu các cách trên mọi cặp `1..100`, và test ngẫu nhiên có seed.

In [22]:
import random

impls = {
    "minimax": winner_minimax,
    "simulation": winner_simulation,
    "parity": winner_parity,
    "rounds": winner_rounds,
    "bit": winner_bit,
    "submit": Solution().losingPlayer,
}

# 1) Ví dụ của đề
for x, y, expected in examples:
    for name, fn in impls.items():
        assert fn(x, y) == expected, (name, x, y)
print("PASS ví dụ đề")

# 2) Biên (tự dựng, đã đối chiếu bằng minimax)
edge = [
    (1, 1, "Bob"),      # y < 4: Alice không đi được
    (1, 3, "Bob"),      # thiếu 1 xu 10
    (1, 4, "Alice"),    # vừa đủ một lượt
    (100, 3, "Bob"),    # xu 75 thừa rất nhiều nhưng thiếu xu 10
    (1, 100, "Alice"),  # xu 10 thừa nhưng chỉ có 1 xu 75 -> T = 1
    (2, 8, "Bob"),      # T = 2
    (100, 100, "Alice"),  # T = min(100, 25) = 25 là số lẻ
]
for x, y, expected in edge:
    for name, fn in impls.items():
        assert fn(x, y) == expected, (name, x, y, expected)
print("PASS biên")

PASS ví dụ đề
PASS biên


In [23]:
# 3) Phản ví dụ: những công thức "gần đúng" nhưng sai
def sai_chi_theo_y(x, y):
    return "Alice" if (y // 4) % 2 else "Bob"

def sai_chi_theo_x(x, y):
    return "Alice" if x % 2 else "Bob"

bad = [(2, 100), (3, 3), (100, 3)]
for x, y in bad:
    print((x, y), "đúng:", Solution().losingPlayer(x, y), "| chỉ theo y:", sai_chi_theo_y(x, y), "| chỉ theo x:", sai_chi_theo_x(x, y))
assert sai_chi_theo_y(2, 100) != Solution().losingPlayer(2, 100)      # T = 2 (chẵn) nhưng y // 4 = 25 (lẻ)
assert sai_chi_theo_x(3, 3) != Solution().losingPlayer(3, 3)          # T = 0 nhưng x = 3 (lẻ)

(2, 100) đúng: Bob | chỉ theo y: Alice | chỉ theo x: Bob
(3, 3) đúng: Bob | chỉ theo y: Bob | chỉ theo x: Alice
(100, 3) đúng: Bob | chỉ theo y: Bob | chỉ theo x: Bob


In [24]:
# 4) Đối chiếu MỌI cặp (x, y) trong 1..100 giữa các cách
count = 0
for x in range(1, 101):
    for y in range(1, 101):
        ref = winner_minimax(x, y)
        for name, fn in impls.items():
            assert fn(x, y) == ref, (name, x, y)
        count += 1
print("PASS đối chiếu toàn bộ", count, "cặp (x, y)")

# 5) Ngẫu nhiên có seed, đúng ràng buộc 1 <= x, y <= 100
random.seed(3222)
for _ in range(3000):
    x, y = random.randint(1, 100), random.randint(1, 100)
    ref = winner_simulation(x, y)
    assert Solution().losingPlayer(x, y) == ref
    assert winner_rounds(x, y) == ref
print("PASS 3000 test ngẫu nhiên")

PASS đối chiếu toàn bộ 10000 cặp (x, y)
PASS 3000 test ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về người thua (bẫy tên hàm `losingPlayer`)

In [25]:
def sai_tra_nguoi_thua(x, y):
    turns = min(x, y // 4)
    return "Bob" if turns % 2 == 1 else "Alice"      # SAI: ngược, vì tên hàm là losingPlayer


print("sai:", sai_tra_nguoi_thua(2, 7), "| đúng:", Solution().losingPlayer(2, 7))

sai: Bob | đúng: Alice


**Vì sao sai:** tên hàm gợi ý "trả người thua" nhưng **đề** ghi rõ trả về người **thắng** (Ví dụ 1 trả `"Alice"` — người thắng). Luôn đối chiếu với ví dụ chứ đừng tin tên hàm. **Cách sửa:** lẻ ⇒ `"Alice"`, chẵn ⇒ `"Bob"` (như code nộp).

### Lỗi 2: Chỉ xét xu `10` (quên rằng còn phải đủ xu `75`)

In [26]:
def sai_quen_xu_75(x, y):
    turns = y // 4                                  # SAI: bỏ qua giới hạn của x
    return "Alice" if turns % 2 else "Bob"


for x, y in [(2, 100), (2, 7), (4, 11)]:
    print((x, y), "sai:", sai_quen_xu_75(x, y), "| đúng:", Solution().losingPlayer(x, y))

(2, 100) sai: Alice | đúng: Bob
(2, 7) sai: Alice | đúng: Alice
(4, 11) sai: Bob | đúng: Bob


**Vì sao sai:** với `x = 2, y = 100`, chỉ có `2` xu `75` nên đúng `2` lượt (chẵn ⇒ Bob thắng), dù `y // 4 = 25` (lẻ ⇒ công thức sai cho Alice). Hai ví dụ của đề trùng kết quả chỉ do may mắn. Luôn dùng `min` của **mọi** nguồn lực. **Cách sửa:** `min(x, y // 4)`.

### Lỗi 3: Dùng `y // 10` hoặc `y // 40` thay cho `y // 4`

In [27]:
def sai_chia_nham(x, y):
    turns = min(x, y // 40)                         # SAI: nhầm 4 xu 10 thành 40 xu
    return "Alice" if turns % 2 else "Bob"


for x, y in [(2, 7), (3, 12), (4, 11)]:
    print((x, y), "sai:", sai_chia_nham(x, y), "| đúng:", Solution().losingPlayer(x, y))
assert sai_chia_nham(2, 7) != Solution().losingPlayer(2, 7)      # T sai = 0 -> "Bob", đúng là "Alice"

(2, 7) sai: Bob | đúng: Alice
(3, 12) sai: Bob | đúng: Alice
(4, 11) sai: Bob | đúng: Bob


**Vì sao sai:** mỗi lượt cần **`4` đồng** xu `10` (giá trị `40`), không phải `40` đồng. Đơn vị bị nhầm giữa "số xu" và "giá trị". **Cách sửa:** viết rõ trong đầu: `y` là **số xu**, mỗi lượt tốn `4` xu.

### Lỗi 4: Mô phỏng nhưng xét sai điều kiện dừng (`or` thay cho `and`)

In [28]:
def sai_or_thay_and(x, y):
    alice_turn = True
    guard = 0
    while (x >= 1 or y >= 4) and guard < 1000:      # SAI: chỉ cần MỘT trong hai đủ thì vẫn lặp
        x -= 1
        y -= 4
        alice_turn = not alice_turn
        guard += 1                                   # chốt chặn tránh vòng lặp vô hạn khi chạy demo
    return "Bob" if alice_turn else "Alice"


print("sai:", sai_or_thay_and(2, 7), "| đúng:", Solution().losingPlayer(2, 7))
print("sai:", sai_or_thay_and(4, 11), "| đúng:", Solution().losingPlayer(4, 11))

sai: Bob | đúng: Alice
sai: Bob | đúng: Bob


**Vì sao sai:** lượt chỉ đi được khi **cả hai** điều kiện `x >= 1` **và** `y >= 4` cùng đúng; dùng `or` làm `x`, `y` âm mà vẫn lặp, đếm sai số lượt. **Cách sửa:** `while x >= 1 and y >= 4`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Hai người luân phiên, mỗi lượt có nước đi cố định**" ⇒ trò chơi xác định: đếm số lượt `T`, rồi xem **chẵn/lẻ**. Người đi lượt cuối thắng (trò "ai không đi được thì thua").
- Đừng vội viết minimax: trước hết **liệt kê các nước đi hợp lệ**; nếu chỉ có một nước thì bài thành mô phỏng.
- Mỗi lượt tiêu hao **nhiều loại tài nguyên** ⇒ số lượt tối đa là `min` của các thương `tài_nguyên // mức_tiêu_hao`.
- Muốn chắc đáp án: viết bản vét cạn nhỏ (minimax có nhớ) rồi `assert` với công thức trên toàn bộ miền nhỏ (ở đây `100 × 100`).
- Đọc kỹ **câu hỏi** (trả người thắng hay thua) và đối chiếu với ví dụ, đừng dựa vào tên hàm.

## 15. Biến thể của bài

1. **Trả về người thua** (đúng như tên hàm gợi ý): chỉ cần đảo kết quả.
2. **Bob đi trước** (hoặc `first` là tham số): đảo vai trò chẵn/lẻ.
3. **Tổng quát hóa**: đồng xu giá trị `v1`, `v2`, mục tiêu `target`; khi đó có thể có **nhiều cách lấy** và cần minimax thật sự (Cách 1).
4. **Số lượt** thay vì tên người: trả `T`.
5. **Tìm cặp `(x, y)` nhỏ nhất** sao cho Bob thắng với `x, y ≥ 1`.
6. Bài cùng họ "trò chơi đơn giản": **Nim Game (292)**, **Bulb Switcher (319)**, **Divisor Game (1025)**, **Stone Game (877)**.

In [29]:
def winner_general(x, y, v1=75, v2=10, target=115, alice_first=True):
    """Minimax tổng quát: x xu giá trị v1, y xu giá trị v2, mỗi lượt lấy tổng đúng target."""
    @lru_cache(maxsize=None)
    def win(x, y):
        for a in range(0, min(x, target // v1) + 1):
            for b in range(0, min(y, target // v2) + 1):
                if a + b > 0 and v1 * a + v2 * b == target and not win(x - a, y - b):
                    return True
        return False

    first_wins = win(x, y)
    if alice_first:
        return "Alice" if first_wins else "Bob"
    return "Bob" if first_wins else "Alice"


def losing_player_true(x, y):
    return "Bob" if Solution().losingPlayer(x, y) == "Alice" else "Alice"


def min_pair_bob_wins():
    for s in range(2, 201):
        for x in range(1, s):
            y = s - x
            if y <= 100 and x <= 100 and Solution().losingPlayer(x, y) == "Bob":
                return (x, y)


print(winner_general(2, 7), winner_general(4, 11))                  # Alice Bob (cùng luật gốc)
print(winner_general(2, 7, alice_first=False))                      # Bob (Bob đi trước)
print(winner_general(3, 3, v1=5, v2=3, target=8))                   # luật khác: có thể chọn nước đi
print(losing_player_true(2, 7), losing_player_true(4, 11))          # Bob Alice
print(min_pair_bob_wins())                                          # (1, 1)
assert winner_general(2, 7) == "Alice" and winner_general(4, 11) == "Bob"
assert min_pair_bob_wins() == (1, 1)
assert all(winner_general(x, y) == Solution().losingPlayer(x, y) for x in range(1, 30) for y in range(1, 30))

Alice Bob
Bob
Alice
Bob Alice
(1, 1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax có nhớ | thử mọi nước đi, "thắng nếu có nước làm đối thủ thua" | `O(x · y)` | `O(x · y)` | luật tổng quát / để **kiểm chứng** |
| 2. Mô phỏng `while` | trừ `(1, 4)` mỗi lượt, đổi lượt | `O(min(x, y/4))` | `O(1)` | dễ hiểu nhất, an toàn |
| 3. `min(x, y // 4)` chẵn/lẻ | đếm số lượt tối đa | `O(1)` | `O(1)` | **nên nộp** |
| 4. Gộp vòng (doocs) | `k` vòng đôi rồi xét nửa vòng cuối | `O(1)` | `O(1)` | tương đương Cách 3, khác góc nhìn |
| 5. Chỉ số bằng bit | `("Bob","Alice")[T & 1]` | `O(1)` | `O(1)` | code golf, ít đọc được |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `min(3, 17 // 4)`, `min(3, 17 // 4) % 2`, `Solution().losingPlayer(3, 17)`, `Solution().losingPlayer(1, 3)`, `Solution().losingPlayer(10, 100)`.

**Bài 2 (tính tay).** Với `x = 5, y = 14`: lập bảng từng lượt (người đi, `x`, `y` trước và sau) cho tới khi có người không đi được, rồi cho biết ai thắng.

**Bài 3 (điền TODO).** Hoàn thành `losing_player_todo(x, y)` theo Cách 2 (mô phỏng).

In [30]:
def losing_player_todo(x, y):
    # TODO 1: đặt alice_turn = True
    # TODO 2: while x >= 1 and y >= 4: trừ x đi 1, y đi 4, đảo alice_turn
    # TODO 3: alice_turn là người KHÔNG đi được -> trả "Bob" nếu alice_turn else "Alice"
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `losing_player_rounds2(x, y)` theo Cách 4 (gộp vòng) nhưng **không** dùng toán tử `and`: dùng `if x > 0: if y >= 4: ...` lồng nhau.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_turns(x, y)` trả về số lượt tối đa `T`. (b) Viết `winner_with_first(x, y, first)` với `first` là `"Alice"` hoặc `"Bob"` (người đi trước), trả về người thắng. (c) Viết `smallest_x_for_alice(y)` trả về `x` nhỏ nhất (`x >= 1`) để Alice thắng với `y` cho trước, hoặc `-1` nếu không tồn tại với `x` tới `100`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `17 // 4 = 4`; hãy tính `min`, rồi xem lẻ/chẵn. `(1, 3)`: `3 < 4` nên Alice không đi nổi lượt đầu. Bài 3: `alice_turn` được đảo **mỗi lần** vòng lặp chạy.
- **Mức 2 (vừa):** Bài 2: `T = min(5, 3) = 3`; liệt kê `(5,14) → (4,10) → (3,6) → (2,2)`; ai tới lượt ở trạng thái `(2, 2)`? Bài 4: `k = min(x // 2, y // 8)` rồi trừ, rồi hai `if` lồng nhau.
- **Mức 3 (sâu):** Bài 5(b): nếu Bob đi trước thì "người đi lượt lẻ" là Bob — đảo kết quả khi `first == "Bob"`. 5(c): thử `x = 1, 2, ..., 100` và dùng `Solution().losingPlayer`; chú ý `y < 4` thì không `x` nào giúp Alice thắng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert min(3, 17 // 4) == 3
assert min(3, 17 // 4) % 2 == 1
assert Solution().losingPlayer(3, 17) == "Alice"
assert Solution().losingPlayer(1, 3) == "Bob"
assert Solution().losingPlayer(10, 100) == "Bob"       # T = min(10, 25) = 10 chẵn

# Bài 2: (5,14) -> (4,10) -> (3,6) -> (2,2); T = 3 nên Alice đi lượt 3, Bob không đi được
x, y = 5, 14
trace = []
alice = True
while x >= 1 and y >= 4:
    trace.append(("Alice" if alice else "Bob", x, y, x - 1, y - 4))
    x, y = x - 1, y - 4
    alice = not alice
assert trace == [("Alice", 5, 14, 4, 10), ("Bob", 4, 10, 3, 6), ("Alice", 3, 6, 2, 2)]
assert Solution().losingPlayer(5, 14) == "Alice"


# Bài 3
def losing_player_todo_ans(x, y):
    alice_turn = True
    while x >= 1 and y >= 4:
        x -= 1
        y -= 4
        alice_turn = not alice_turn
    return "Bob" if alice_turn else "Alice"


for x, y, expected in examples:
    assert losing_player_todo_ans(x, y) == expected


# Bài 4
def losing_player_rounds2(x, y):
    k = min(x // 2, y // 8)
    x -= 2 * k
    y -= 8 * k
    if x > 0:
        if y >= 4:
            return "Alice"
    return "Bob"


for x in range(1, 40):
    for y in range(1, 60):
        assert losing_player_rounds2(x, y) == Solution().losingPlayer(x, y)


# Bài 5
def count_turns(x, y):
    return min(x, y // 4)


def winner_with_first(x, y, first):
    t = count_turns(x, y)
    first_wins = t % 2 == 1
    if first == "Alice":
        return "Alice" if first_wins else "Bob"
    return "Bob" if first_wins else "Alice"


def smallest_x_for_alice(y):
    for x in range(1, 101):
        if Solution().losingPlayer(x, y) == "Alice":
            return x
    return -1


assert count_turns(4, 11) == 2 and count_turns(2, 7) == 1
assert winner_with_first(2, 7, "Alice") == "Alice" and winner_with_first(2, 7, "Bob") == "Bob"
assert winner_with_first(4, 11, "Bob") == "Alice"
assert winner_with_first(5, 14, "Alice") == winner_general(5, 14)
assert smallest_x_for_alice(7) == 1 and smallest_x_for_alice(3) == -1 and smallest_x_for_alice(8) == 1
assert smallest_x_for_alice(100) == 1
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm số lượt tối đa rồi xét chẵn/lẻ — lẻ ⇒ Alice thắng)

```python
class Solution:
    def losingPlayer(self, x: int, y: int) -> str:
        turns = min(x, y // 4)          # mỗi lượt: 1 xu 75 + 4 xu 10
        return "Alice" if turns % 2 == 1 else "Bob"
```

**Checklist**

- [ ] Đã chứng minh/ kiểm tra rằng `115 = 75 + 4 · 10` là **cách lấy duy nhất**?
- [ ] Số lượt `T = min(x, y // 4)` — có đủ **cả hai** nguồn lực?
- [ ] `T` lẻ ⇒ `"Alice"`, `T` chẵn (kể cả `0`) ⇒ `"Bob"` — không bị đảo ngược?
- [ ] Trả về người **thắng** (không phải người thua dù tên hàm là `losingPlayer`)?
- [ ] Đã thử `(2, 7)` ⇒ `"Alice"`, `(4, 11)` ⇒ `"Bob"`, `(1, 3)` ⇒ `"Bob"` và `(1, 4)` ⇒ `"Alice"`?
- [ ] Skeleton đúng `class Solution` / `def losingPlayer(self, x: int, y: int) -> str`?